# ENSEMBLE UPLIFT_RF_CIT + INT_RF — Full-train inference + Feature importance

Notebook này đã bỏ toàn bộ phần huấn luyện K-fold.

Luồng chạy cuối cùng:

1. Load `df_feature.csv`, `df_target.csv`, `test_df_feature.csv`.
2. Train 5 base model trên toàn bộ tập train:
   - 3 cấu hình `UPLIFT_RF_CIT`.
   - 2 cấu hình `INT_RF`.
3. Inference từng model trên toàn bộ `test_df_feature.csv`.
4. In Qini trên toàn bộ train cho từng model với 3 cột: `qini_cash`, `qini_card`, `qini_sum`.
5. Xuất `submission.csv`.
6. Xuất folder `feature_importance` gồm 5 file importance chi tiết và 1 file tổng hợp.

Công thức uplift dùng trong notebook:

`uplift_cash(x) = P(cash_sign_30d = 1 | treatment = Cash, x) - P(cash_sign_30d = 1 | treatment = Card, x)`

`uplift_card(x) = P(card_sign_30d = 1 | treatment = Card, x) - P(card_sign_30d = 1 | treatment = Cash, x)`

`qini_sum = qini_cash + qini_card`


In [6]:
import sys
import subprocess

packages = {
    "causalml": "causalml==0.16.0",
    "shap": "shap",
    "lime": "lime",
}

for module_name, package_name in packages.items():
    try:
        __import__(module_name)
        print(f"{module_name} is already installed.")
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        print(f"{package_name} installed.")


causalml is already installed.
shap is already installed.
lime is already installed.


## 1. Imports và cấu hình

In [7]:
import gc
import os
import re
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

import numpy as np
import pandas as pd
from IPython.display import display

import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.ensemble import RandomForestClassifier
from causalml.inference.tree import UpliftRandomForestClassifier

warnings.filterwarnings("ignore")

SEED = 42
N_JOBS = 4
CAUSALML_CRITERION = "CIT"

TARGET_CASH = "cash_sign_30d"
TARGET_CARD = "card_sign_30d"
TREAT_COL_BINARY = "treatment_cash"
TREAT_COL_RAW = "treatment"
ID_CANDIDATES = ["id", "ID", "row_id", "customer_id", "Customer_ID", "client_id", "cli_id", "app_id", "application_id"]

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
CP = WORKING_DIR / "cp"
FEATURE_IMPORTANCE_DIR = CP / "feature_importance"
CP.mkdir(parents=True, exist_ok=True)
FEATURE_IMPORTANCE_DIR.mkdir(parents=True, exist_ok=True)

UPLIFT_RF_CIT_CONFIGS = [
    {"name": "CFG028_UPLIFT_RF_CIT", "n_estimators": 80, "max_depth": 8, "min_samples_leaf": 120, "min_samples_treatment": 20, "max_features_cap": 32},
    {"name": "CFG009_UPLIFT_RF_CIT", "n_estimators": 120, "max_depth": 5, "min_samples_leaf": 80, "min_samples_treatment": 30, "max_features_cap": 32},
    {"name": "CFG011_UPLIFT_RF_CIT", "n_estimators": 120, "max_depth": 5, "min_samples_leaf": 160, "min_samples_treatment": 40, "max_features_cap": 32},
]

INT_RF_CONFIGS = [
    {"name": "INT_RF_RS_027", "n_estimators": 560, "max_depth": 6, "min_samples_leaf": 80, "min_samples_split": 50, "max_features": "log2", "criterion": "entropy", "class_weight": "balanced", "bootstrap": True, "max_samples": None, "use_sample_weight": False, "pos_weight_multiplier": 1.0},
    {"name": "INT_RF_RS_028", "n_estimators": 420, "max_depth": 6, "min_samples_leaf": 30, "min_samples_split": 2, "max_features": "log2", "criterion": "gini", "class_weight": "balanced", "bootstrap": True, "max_samples": None, "use_sample_weight": False, "pos_weight_multiplier": 1.0},
]

ENSEMBLE_NAME = "ENSEMBLE_3_UPLIFT_RF_CIT_2_INT_RF"

IMPORTANCE_SAMPLE_SIZE = 800
LIME_SAMPLE_SIZE = 30
SHAP_BACKGROUND_SIZE = 40
SHAP_SAMPLE_SIZE = 80
SHAP_NSAMPLES = 129
PERMUTATION_REPEATS = 3

print("=" * 80)
print("IMPORTS DONE")
print(f"SEED={SEED} | N_JOBS={N_JOBS}")
print("CP:", CP)
print("TRAINING MODE: full-train only, no K-fold")
print("UPLIFT_RF_CIT_CONFIGS:")
for cfg in UPLIFT_RF_CIT_CONFIGS:
    print(" ", cfg)
print("INT_RF_CONFIGS:")
for cfg in INT_RF_CONFIGS:
    print(" ", cfg)
print("ENSEMBLE:", ENSEMBLE_NAME)
print("=" * 80)


IMPORTS DONE
SEED=42 | N_JOBS=4
CP: /kaggle/working/cp
TRAINING MODE: full-train only, no K-fold
UPLIFT_RF_CIT_CONFIGS:
  {'name': 'CFG028_UPLIFT_RF_CIT', 'n_estimators': 80, 'max_depth': 8, 'min_samples_leaf': 120, 'min_samples_treatment': 20, 'max_features_cap': 32}
  {'name': 'CFG009_UPLIFT_RF_CIT', 'n_estimators': 120, 'max_depth': 5, 'min_samples_leaf': 80, 'min_samples_treatment': 30, 'max_features_cap': 32}
  {'name': 'CFG011_UPLIFT_RF_CIT', 'n_estimators': 120, 'max_depth': 5, 'min_samples_leaf': 160, 'min_samples_treatment': 40, 'max_features_cap': 32}
INT_RF_CONFIGS:
  {'name': 'INT_RF_RS_027', 'n_estimators': 560, 'max_depth': 6, 'min_samples_leaf': 80, 'min_samples_split': 50, 'max_features': 'log2', 'criterion': 'entropy', 'class_weight': 'balanced', 'bootstrap': True, 'max_samples': None, 'use_sample_weight': False, 'pos_weight_multiplier': 1.0}
  {'name': 'INT_RF_RS_028', 'n_estimators': 420, 'max_depth': 6, 'min_samples_leaf': 30, 'min_samples_split': 2, 'max_features':

## 2. Load dữ liệu

In [8]:
def first_existing_path(paths):
    for p in paths:
        if Path(p).exists():
            return str(p)
    return str(paths[0])

DATA_DIR = Path("/kaggle/input/datasets/tmtrnhelloworld/feature-eng-data-daz")

DF_FEATURE_CANDIDATES = [
    DATA_DIR / "df_feature.csv",
    Path("/mnt/data/df_feature.csv"),
    Path("df_feature.csv"),
]

DF_TARGET_CANDIDATES = [
    DATA_DIR / "df_target.csv",
    Path("/mnt/data/df_target.csv"),
    Path("df_target.csv"),
]

TEST_FEATURE_CANDIDATES = [
    DATA_DIR / "test_df_feature.csv",
    Path("/mnt/data/test_df_feature.csv"),
    Path("test_df_feature.csv"),
]

df_feature_path = first_existing_path(DF_FEATURE_CANDIDATES)
df_target_path = first_existing_path(DF_TARGET_CANDIDATES)
test_df_feature_path = first_existing_path(TEST_FEATURE_CANDIDATES)

df_feature = pd.read_csv(df_feature_path)
df_target = pd.read_csv(df_target_path)
test_df_feature = pd.read_csv(test_df_feature_path)

print("=" * 80)
print("DATA LOADED")
print("df_feature_path      :", df_feature_path)
print("df_target_path       :", df_target_path)
print("test_df_feature_path :", test_df_feature_path)
print(f"df_feature       : {df_feature.shape}")
print(f"df_target        : {df_target.shape}")
print(f"test_df_feature  : {test_df_feature.shape}")
print("CP:", CP)
print("=" * 80)


DATA LOADED
df_feature_path      : /kaggle/input/datasets/tmtrnhelloworld/feature-eng-data-daz/df_feature.csv
df_target_path       : /kaggle/input/datasets/tmtrnhelloworld/feature-eng-data-daz/df_target.csv
test_df_feature_path : /kaggle/input/datasets/tmtrnhelloworld/feature-eng-data-daz/test_df_feature.csv
df_feature       : (136992, 63)
df_target        : (136992, 2)
test_df_feature  : (34080, 63)
CP: /kaggle/working/cp


## 3. Chuẩn bị feature và target

In [9]:
# ============================================================
# 3. FEATURE PREPARATION HELPERS
# ============================================================
FEATURE_DROP_PRIORITY_LIST = [
    "cnt_month_over3_credit_card_6m",
    "cnt_connected_tls",
    "cnt_earlyterminted_pos",
    "min_sign_vol",
    "cli_contact_region_population_grp",
    "cnt_inbound_tls_12m",
    "cnt_bank_inst_appl_12m",
    "f_approve_last_process",
    "worst_credit_status",
    "cli_contact_region_level",
    "cli_education",
    "f_ever_default",
    "cli_age",
    "cli_job",
    "cnt_earlyterminated_cash",
    "amt_terminated_loan_12m",
    "cnt_noinsurance_loan_24m",
    "cnt_report_bank_6m",
    "ratio_conn_tls_endmonth_12m",
    "ratio_conn_tls_weekend_12m",
    "cli_marital_status",
    "avg_sign_vol",
    "cnt_terminated_loan_12m",
    "cnt_fail_tls_3m",
    "last_offer_limit",
    "cli_emp_period",
    "cnt_fi_inst_appl_12m",
    "cnt_total_payment",
    "risk_grp",
    "cnt_lender_has2pay_last_24m",
    "cnt_month_status1",
    "last_process_channel",
    "cnt_month_from_last_appl",
    "cnt_report_institution_12m",
]

# None -> baseline, không drop feature nào.
COL_IDX = None


def normalize_feature_drop_selection(selection):
    if selection is None:
        return [], []

    def _expand_one(item):
        if isinstance(item, str):
            s = item.strip()
            if not s:
                return []
            if re.fullmatch(r"\d+", s):
                return [int(s)]
            m = re.fullmatch(r"(\d+)\s*:\s*(\d+)", s)
            if m:
                return list(range(int(m.group(1)), int(m.group(2))))
            return [s]
        if isinstance(item, (int, np.integer)):
            return [int(item)]
        if isinstance(item, range):
            return list(item)
        if isinstance(item, slice):
            start = 0 if item.start is None else item.start
            stop = len(FEATURE_DROP_PRIORITY_LIST) if item.stop is None else item.stop
            step = 1 if item.step is None else item.step
            return list(range(start, stop, step))
        raise TypeError("COL_IDX không hợp lệ.")

    if isinstance(selection, (list, tuple, set, range, np.ndarray, pd.Series)):
        raw_items = []
        for item in list(selection):
            raw_items.extend(_expand_one(item))
    else:
        raw_items = _expand_one(selection)

    drop_ids = []
    drop_base_cols = []
    seen_cols = set()

    for item in raw_items:
        if isinstance(item, (int, np.integer)):
            idx = int(item)
            if idx < 0 or idx >= len(FEATURE_DROP_PRIORITY_LIST):
                raise IndexError(f"COL_IDX={idx} ngoài range 0..{len(FEATURE_DROP_PRIORITY_LIST)-1}")
            base_col = FEATURE_DROP_PRIORITY_LIST[idx]
        else:
            base_col = str(item).strip()
            idx = FEATURE_DROP_PRIORITY_LIST.index(base_col) if base_col in FEATURE_DROP_PRIORITY_LIST else None
            if idx is None:
                print(f"WARNING: '{base_col}' không nằm trong FEATURE_DROP_PRIORITY_LIST, vẫn sẽ thử match trong feature.")

        if base_col not in seen_cols:
            drop_ids.append(idx)
            drop_base_cols.append(base_col)
            seen_cols.add(base_col)

    return drop_ids, drop_base_cols


def match_base_feature_columns(columns, base_col):
    if base_col is None:
        return []
    return [c for c in columns if c == base_col or c.startswith(f"{base_col}_")]


def make_id_series(source_df, n_rows=None, id_col=None):
    if n_rows is None:
        n_rows = len(source_df)
    if id_col is not None and id_col in source_df.columns:
        return source_df[id_col].reset_index(drop=True).rename(id_col)
    return pd.Series(np.arange(n_rows), name="row_id")


def encode_treatment_cash(feature_df, target_df=None):
    source_df = feature_df
    source_name = "feature"

    if TREAT_COL_BINARY not in source_df.columns and TREAT_COL_RAW not in source_df.columns and target_df is not None:
        source_df = target_df
        source_name = "target"

    if TREAT_COL_BINARY in source_df.columns:
        return source_df[TREAT_COL_BINARY].astype(int).to_numpy(), TREAT_COL_BINARY, source_name

    if TREAT_COL_RAW not in source_df.columns:
        raise ValueError("Thiếu cột treatment_cash hoặc treatment trong feature/target.")

    s = source_df[TREAT_COL_RAW]
    if pd.api.types.is_numeric_dtype(s):
        vals = s.astype(int).to_numpy()
        unique_vals = set(pd.Series(vals).dropna().unique())
        if unique_vals <= {0, 1}:
            return vals, TREAT_COL_RAW, source_name
        raise ValueError(f"Cột treatment dạng số nhưng không phải binary 0/1. Unique mẫu: {sorted(list(unique_vals))[:10]}")

    st = s.astype(str).str.strip().str.lower()
    mapped = np.where(st.str.contains("cash"), 1, np.where(st.str.contains("card"), 0, np.nan))
    if np.isnan(mapped).any():
        bad = sorted(st[pd.isna(mapped)].unique()[:10])
        raise ValueError(f"Không encode được treatment values: {bad}")

    return mapped.astype(int), TREAT_COL_RAW, source_name


def get_feature_columns(train_feature_df, test_feature_df=None):
    feature_drop_idxs, feature_drop_base_cols = normalize_feature_drop_selection(COL_IDX)

    drop_cols = []
    for base_col in feature_drop_base_cols:
        drop_cols.extend(match_base_feature_columns(train_feature_df.columns, base_col))
    drop_cols = list(dict.fromkeys(drop_cols))

    train_id_col = next((col for col in ID_CANDIDATES if col in train_feature_df.columns), None)
    test_id_col = None
    if test_feature_df is not None:
        test_id_col = next((col for col in ID_CANDIDATES if col in test_feature_df.columns), None)

    index_like_cols = [c for c in train_feature_df.columns if c.lower().startswith("unnamed") or c.lower() == "index"]
    exclude_cols = set(drop_cols + index_like_cols + [TARGET_CASH, TARGET_CARD, TREAT_COL_BINARY, TREAT_COL_RAW])
    if train_id_col is not None:
        exclude_cols.add(train_id_col)

    feat_cols = [col for col in train_feature_df.columns if col not in exclude_cols]
    return feat_cols, drop_cols, feature_drop_idxs, feature_drop_base_cols, train_id_col, test_id_col


def make_design_matrices(fit_feature_df, transform_feature_dfs, feat_cols):
    """
    Fit preprocessing on fit_feature_df, then transform validation/test frames.
    - Numeric columns: median imputation.
    - Categorical/object columns: one-hot encoding, aligned to training dummies.
    """
    fit_raw = fit_feature_df.reindex(columns=feat_cols, fill_value=0).copy()
    transform_raws = [df.reindex(columns=feat_cols, fill_value=0).copy() for df in transform_feature_dfs]

    cat_cols = fit_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in fit_raw.columns if c not in cat_cols]

    if num_cols:
        fit_num = fit_raw[num_cols].apply(pd.to_numeric, errors="coerce")
        med = fit_num.median(axis=0).fillna(0)
        fit_num = fit_num.fillna(med)

        transformed_nums = []
        for raw in transform_raws:
            tnum = raw[num_cols].apply(pd.to_numeric, errors="coerce").fillna(med)
            transformed_nums.append(tnum)
    else:
        fit_num = pd.DataFrame(index=fit_raw.index)
        transformed_nums = [pd.DataFrame(index=raw.index) for raw in transform_raws]

    if cat_cols:
        fit_cat = pd.get_dummies(fit_raw[cat_cols].astype("object").fillna("__MISSING__"), dummy_na=False)
        transformed_cats = []
        for raw in transform_raws:
            tcat = pd.get_dummies(raw[cat_cols].astype("object").fillna("__MISSING__"), dummy_na=False)
            tcat = tcat.reindex(columns=fit_cat.columns, fill_value=0)
            transformed_cats.append(tcat)
    else:
        fit_cat = pd.DataFrame(index=fit_raw.index)
        transformed_cats = [pd.DataFrame(index=raw.index) for raw in transform_raws]

    X_fit_df = pd.concat([fit_num.reset_index(drop=True), fit_cat.reset_index(drop=True)], axis=1)
    X_fit = X_fit_df.to_numpy(dtype=np.float32)

    X_transforms = []
    for tnum, tcat in zip(transformed_nums, transformed_cats):
        X_df = pd.concat([tnum.reset_index(drop=True), tcat.reset_index(drop=True)], axis=1)
        X_df = X_df.reindex(columns=X_fit_df.columns, fill_value=0)
        X_transforms.append(X_df.to_numpy(dtype=np.float32))

    return X_fit, X_transforms, X_fit_df.columns.tolist()


assert len(df_feature) == len(df_target), "df_feature và df_target khác số dòng."
for col in [TARGET_CASH, TARGET_CARD]:
    assert col in df_target.columns, f"Thiếu target column: {col}"

W_all, TREAT_COL_TRAIN, TREAT_SOURCE_TRAIN = encode_treatment_cash(df_feature, df_target)
Y_cash_all = df_target[TARGET_CASH].astype(int).to_numpy()
Y_card_all = df_target[TARGET_CARD].astype(int).to_numpy()

FEAT_COLS, DROP_COLS_IN_TRAIN, FEATURE_DROP_IDXS, FEATURE_DROP_BASE_COLS, train_id_col, test_id_col = get_feature_columns(df_feature, test_df_feature)
train_id = make_id_series(df_feature, id_col=train_id_col)
test_id = make_id_series(test_df_feature, id_col=test_id_col)

FORBIDDEN_FEATURE_COLS = {TARGET_CASH, TARGET_CARD, TREAT_COL_BINARY, TREAT_COL_RAW}
if train_id_col is not None:
    FORBIDDEN_FEATURE_COLS.add(train_id_col)
forbidden_present = sorted([c for c in FEAT_COLS if c in FORBIDDEN_FEATURE_COLS])
assert not forbidden_present, f"DATA LEAKAGE: các cột không được phép đang nằm trong FEAT_COLS: {forbidden_present}"

feature_drop_audit = pd.DataFrame({
    "base_feature": FEATURE_DROP_PRIORITY_LIST,
    "n_matched_train_cols": [len(match_base_feature_columns(df_feature.columns, c)) for c in FEATURE_DROP_PRIORITY_LIST],
    "matched_train_cols": [match_base_feature_columns(df_feature.columns, c) for c in FEATURE_DROP_PRIORITY_LIST],
})
feature_drop_audit.to_csv(CP / "feature_drop_audit.csv", index=False)

print("=" * 80)
if not FEATURE_DROP_BASE_COLS:
    print("FEATURE DROPPING: baseline, không drop feature nào")
else:
    print(f"FEATURE DROPPING: COL_IDX={COL_IDX}")
    print(f"Drop IDs      : {FEATURE_DROP_IDXS}")
    print(f"Base features : {FEATURE_DROP_BASE_COLS}")
    print(f"Dropped train feature columns ({len(DROP_COLS_IN_TRAIN)}): {DROP_COLS_IN_TRAIN}")
print("=" * 80)
print(f"Train N={len(df_feature):,} | Test N={len(test_df_feature):,} | raw feature columns={len(FEAT_COLS):,}")
print(f"Train treatment source: {TREAT_SOURCE_TRAIN}.{TREAT_COL_TRAIN}")
print(f"Train Treatment Cash: {W_all.sum():,} ({W_all.mean():.2%}) | Card: {(1-W_all).sum():,} ({(1-W_all).mean():.2%})")
print(f"Train cash target rate: {Y_cash_all.mean():.4%} | Train card target rate: {Y_card_all.mean():.4%}")
print("Saved:", CP / "feature_drop_audit.csv")
print("=" * 80)
display(feature_drop_audit)


FEATURE DROPPING: baseline, không drop feature nào
Train N=136,992 | Test N=34,080 | raw feature columns=62
Train treatment source: feature.treatment_cash
Train Treatment Cash: 75,612 (55.19%) | Card: 61,380 (44.81%)
Train cash target rate: 5.9186% | Train card target rate: 2.7111%
Saved: /kaggle/working/cp/feature_drop_audit.csv


,base_feature,n_matched_train_cols,matched_train_cols
0,cnt_month_over3_credit_card_6m,1,[cnt_month_over3_credit_card_6m]
1,cnt_connected_tls,1,[cnt_connected_tls]
2,cnt_earlyterminted_pos,1,[cnt_earlyterminted_pos]
3,min_sign_vol,1,[min_sign_vol]
4,cli_contact_region_population_grp,3,"[cli_contact_region_population_grp_RURAL, cli_..."
5,cnt_inbound_tls_12m,1,[cnt_inbound_tls_12m]
6,cnt_bank_inst_appl_12m,1,[cnt_bank_inst_appl_12m]
7,f_approve_last_process,1,[f_approve_last_process]
8,worst_credit_status,1,[worst_credit_status]
9,cli_contact_region_level,2,"[cli_contact_region_level_North, cli_contact_r..."


## 4. Qini metric

In [10]:
_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def qini_curve(y, w_event, score):
    y = np.asarray(y, dtype=float)
    w_event = np.asarray(w_event, dtype=int)
    score = np.asarray(score, dtype=float)
    order = np.argsort(-score, kind="mergesort")
    y_s = y[order]
    w_s = w_event[order]
    n_t = max(int(w_event.sum()), 1)
    n_c = max(len(w_event) - n_t, 1)
    q = np.cumsum(y_s * w_s) / n_t - np.cumsum(y_s * (1 - w_s)) / n_c
    x = np.arange(1, len(y) + 1) / len(y)
    return np.r_[0.0, x], np.r_[0.0, q]


def oracle_score(y, w_event):
    y = np.asarray(y, dtype=int)
    w_event = np.asarray(w_event, dtype=int)
    return w_event * y - (1 - w_event) * y


def qini_metrics(y, w_event, score):
    x, q = qini_curve(y, w_event, score)
    auc_model = float(_trapz(q, x))
    ate = float(q[-1])
    auc_random = ate * 0.5
    _, q_perfect = qini_curve(y, w_event, oracle_score(y, w_event))
    auc_perfect = float(_trapz(q_perfect, x))
    qini_diff = auc_model - auc_random
    denom = auc_perfect - auc_random
    qini_norm = qini_diff / denom if abs(denom) > 1e-12 else np.nan
    qini_ratio = auc_model / auc_random if abs(auc_random) > 1e-12 else np.nan
    return {
        "ate": ate,
        "auc_model": auc_model,
        "auc_random": auc_random,
        "auc_perfect": auc_perfect,
        "qini_diff": qini_diff,
        "qini_norm": qini_norm,
        "qini_ratio": qini_ratio,
    }


print("QINI FUNCTIONS READY")


QINI FUNCTIONS READY


## 5. UPLIFT_RF_CIT helpers

In [11]:
# ============================================================
# 5. UPLIFT_RF_CIT HELPERS
# ============================================================
def get_treatment_labels(w_event):
    return np.where(np.asarray(w_event) == 1, "treatment", "control")


def make_causalml_rf(params, seed, n_features):
    max_features = int(max(1, min(int(params["max_features_cap"]), n_features)))
    base_kwargs = dict(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        min_samples_treatment=params["min_samples_treatment"],
        max_features=max_features,
        evaluationFunction=CAUSALML_CRITERION,
        control_name="control",
    )

    for extra_kwargs in [{"n_jobs": N_JOBS, "random_state": seed}, {"random_state": seed}, {"n_jobs": N_JOBS}, {}]:
        try:
            return UpliftRandomForestClassifier(**base_kwargs, **extra_kwargs)
        except TypeError:
            continue

    return UpliftRandomForestClassifier(**base_kwargs)


def extract_uplift_prediction(pred_raw):
    if isinstance(pred_raw, pd.DataFrame):
        num = pred_raw.select_dtypes(include=[np.number])
        for c in ["uplift", "treatment_effect", "maxTreatmentEffect", "recommended_treatment_effect"]:
            if c in num.columns:
                return num[c].to_numpy(dtype=float)
        return num.iloc[:, -1].to_numpy(dtype=float)

    arr = np.asarray(pred_raw)
    if arr.ndim == 1:
        return arr.astype(float)
    if arr.ndim >= 2 and arr.shape[1] >= 2:
        return (arr[:, 1] - arr[:, 0]).astype(float)
    return arr[:, 0].astype(float)


def train_predict_uplift_rf_cit(params, target_name, X_fit, y_fit, w_event_fit, X_pred):
    target_offset = 101 if target_name == "cash" else 202
    model_seed = SEED + target_offset + int(params["n_estimators"])
    model = make_causalml_rf(params=params, seed=model_seed, n_features=X_fit.shape[1])

    treatment_labels_fit = get_treatment_labels(w_event_fit)
    model.fit(X_fit, treatment=treatment_labels_fit, y=y_fit)

    uplift_pred = extract_uplift_prediction(model.predict(X_pred))

    # CausalML Uplift RF trả về uplift score. Để xuất xác suất cho test,
    # dùng control base-rate trên tập fit rồi cộng uplift, sau đó clip về [0, 1].
    control_mask = np.asarray(w_event_fit) == 0
    control_rate = float(np.mean(y_fit[control_mask])) if control_mask.sum() > 0 else float(np.mean(y_fit))
    prob_event = np.clip(control_rate + uplift_pred, 0.0, 1.0)

    return {
        "model": model,
        "uplift": uplift_pred.astype(float),
        "prob_event": prob_event.astype(float),
        "seed": model_seed,
    }


print("UPLIFT_RF_CIT HELPERS READY")


UPLIFT_RF_CIT HELPERS READY


## 6. INT_RF helpers

In [12]:
# ============================================================
# 6. INT_RF HELPERS
# ============================================================
class ConstantClassifier:
    def __init__(self, value):
        self.value = float(value)
        self.classes_ = np.array([0, 1])

    def fit(self, X, y, sample_weight=None):
        return self

    def predict_proba(self, X):
        p = np.full(len(X), self.value, dtype=float)
        p = np.clip(p, 0.0, 1.0)
        return np.c_[1 - p, p]


def make_rf_classifier(params, seed=SEED):
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        min_samples_split=params["min_samples_split"],
        max_features=params["max_features"],
        criterion=params["criterion"],
        class_weight=params["class_weight"],
        bootstrap=params["bootstrap"],
        max_samples=params["max_samples"],
        n_jobs=N_JOBS,
        random_state=seed,
    )


def class_weights(y, pos_weight_multiplier=1.0):
    y = np.asarray(y, dtype=int)
    pos = max(int(y.sum()), 1)
    neg = max(len(y) - pos, 1)
    pos_weight = (neg / pos) * float(pos_weight_multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(np.float32)


def fit_classifier(X_fit, y_fit, params, seed=SEED):
    y_fit = np.asarray(y_fit, dtype=int)
    if len(y_fit) == 0 or len(np.unique(y_fit)) < 2:
        return ConstantClassifier(float(y_fit.mean()) if len(y_fit) else 0.0)

    model = make_rf_classifier(params=params, seed=seed)
    sample_weight = class_weights(y_fit, params["pos_weight_multiplier"]) if params["use_sample_weight"] else None
    model.fit(X_fit, y_fit, sample_weight=sample_weight)
    return model


def predict_classifier(model, X_pred):
    proba = np.asarray(model.predict_proba(X_pred))
    if proba.ndim == 1:
        return np.clip(proba.astype(float), 0.0, 1.0)
    if proba.shape[1] == 1:
        classes = getattr(model, "classes_", np.array([1]))
        return np.full(len(X_pred), float(classes[0] == 1))
    classes = list(getattr(model, "classes_", [0, 1]))
    idx = classes.index(1) if 1 in classes else -1
    return proba[:, idx].astype(float)


def make_interaction_matrix(X_base, w_event):
    w_col = np.asarray(w_event, dtype=np.float32)[:, None]
    return np.c_[X_base, w_col, X_base * w_col]


def make_event_control_matrices(X_base):
    n = len(X_base)
    X_event = np.c_[X_base, np.ones((n, 1), dtype=np.float32), X_base]
    X_control = np.c_[X_base, np.zeros((n, 1), dtype=np.float32), np.zeros_like(X_base)]
    return X_event, X_control


def train_predict_int_rf(params, target_name, X_fit, y_fit, w_event_fit, X_pred):
    seed_offset = 303 if target_name == "cash" else 404
    model_seed = SEED + seed_offset + int(params["n_estimators"])

    X_aug_fit = make_interaction_matrix(X_fit, w_event_fit)
    model = fit_classifier(X_aug_fit, y_fit, params=params, seed=model_seed)

    X_pred_event, X_pred_control = make_event_control_matrices(X_pred)
    p_event = predict_classifier(model, X_pred_event)
    p_control = predict_classifier(model, X_pred_control)
    uplift_pred = p_event - p_control

    return {
        "model": model,
        "uplift": uplift_pred.astype(float),
        "prob_event": np.clip(p_event, 0.0, 1.0).astype(float),
        "seed": model_seed,
    }


print("INT_RF HELPERS READY")


INT_RF HELPERS READY


## 7. Train ensemble

In [13]:
# ============================================================
# 7. ENSEMBLE TRAIN/PREDICT FUNCTION
# ============================================================
def train_ensemble_predict(X_fit, y_cash_fit, y_card_fit, W_fit, X_pred, phase_label=""):
    """
    Return:
    - uplift_cash/uplift_card: dùng để tính Qini.
    - prob_cash/prob_card: dùng để xuất test_df_target.
    """
    pred_store = {
        "cash": {"uplift": {}, "prob_event": {}},
        "card": {"uplift": {}, "prob_event": {}},
    }
    model_store = {}
    log_rows = []

    targets = {
        "cash": {
            "y": np.asarray(y_cash_fit, dtype=int),
            "w_event": (np.asarray(W_fit) == 1).astype(int),  # Cash là treatment event
            "family_label": "CASH",
        },
        "card": {
            "y": np.asarray(y_card_fit, dtype=int),
            "w_event": (np.asarray(W_fit) == 0).astype(int),  # Card là treatment event
            "family_label": "CARD",
        },
    }

    t_all = time.time()

    for cfg in UPLIFT_RF_CIT_CONFIGS:
        model_store[cfg["name"]] = {}
        print("\n" + "=" * 80)
        print(f"{phase_label} START {cfg['name']}")
        print(cfg)
        print("=" * 80)

        for target_name, info in targets.items():
            t0 = time.time()
            out = train_predict_uplift_rf_cit(
                params=cfg,
                target_name=target_name,
                X_fit=X_fit,
                y_fit=info["y"],
                w_event_fit=info["w_event"],
                X_pred=X_pred,
            )
            pred_store[target_name]["uplift"][cfg["name"]] = out["uplift"]
            pred_store[target_name]["prob_event"][cfg["name"]] = out["prob_event"]
            model_store[cfg["name"]][target_name] = out["model"]
            log_rows.append({
                "phase": phase_label,
                "model": cfg["name"],
                "model_family": "UPLIFT_RF_CIT",
                "target": target_name,
                "seed": out["seed"],
                "minutes": (time.time() - t0) / 60,
            })
            print(f"{phase_label} finished {cfg['name']} target={target_name.upper()} | minutes={(time.time() - t0) / 60:.4f}")
            gc.collect()

    for cfg in INT_RF_CONFIGS:
        model_store[cfg["name"]] = {}
        print("\n" + "=" * 80)
        print(f"{phase_label} START {cfg['name']}")
        print(cfg)
        print("=" * 80)

        for target_name, info in targets.items():
            t0 = time.time()
            out = train_predict_int_rf(
                params=cfg,
                target_name=target_name,
                X_fit=X_fit,
                y_fit=info["y"],
                w_event_fit=info["w_event"],
                X_pred=X_pred,
            )
            pred_store[target_name]["uplift"][cfg["name"]] = out["uplift"]
            pred_store[target_name]["prob_event"][cfg["name"]] = out["prob_event"]
            model_store[cfg["name"]][target_name] = out["model"]
            log_rows.append({
                "phase": phase_label,
                "model": cfg["name"],
                "model_family": "INT_RF",
                "target": target_name,
                "seed": out["seed"],
                "minutes": (time.time() - t0) / 60,
            })
            print(f"{phase_label} finished {cfg['name']} target={target_name.upper()} | minutes={(time.time() - t0) / 60:.4f}")
            gc.collect()

    base_model_names = list(pred_store["cash"]["uplift"].keys())

    def avg(target_name, key):
        return np.mean(
            np.column_stack([np.asarray(pred_store[target_name][key][name], dtype=float) for name in base_model_names]),
            axis=1,
        )

    result = {
        "uplift_cash": avg("cash", "uplift"),
        "uplift_card": avg("card", "uplift"),
        "prob_cash": np.clip(avg("cash", "prob_event"), 0.0, 1.0),
        "prob_card": np.clip(avg("card", "prob_event"), 0.0, 1.0),
        "base_model_names": base_model_names,
        "base_predictions": pred_store,
        "models": model_store,
        "log_rows": log_rows,
        "minutes_total": (time.time() - t_all) / 60,
    }

    print("\n" + "=" * 80)
    print(f"{phase_label} ENSEMBLE DONE | minutes_total={result['minutes_total']:.4f}")
    print("Base models:", base_model_names)
    print("=" * 80)

    return result


print("ENSEMBLE FUNCTION READY")


ENSEMBLE FUNCTION READY


## 8. LIME / SHAP / Permutation importance

In [14]:
def get_base_model_family(model_name):
    uplift_names = {cfg["name"] for cfg in UPLIFT_RF_CIT_CONFIGS}
    int_names = {cfg["name"] for cfg in INT_RF_CONFIGS}
    if model_name in uplift_names:
        return "UPLIFT_RF_CIT"
    if model_name in int_names:
        return "INT_RF"
    raise ValueError(f"Unknown model_name={model_name}")


def predict_trained_model_uplift(trained_model, model_family, X):
    if model_family == "UPLIFT_RF_CIT":
        return extract_uplift_prediction(trained_model.predict(X)).astype(float)
    if model_family == "INT_RF":
        X_event, X_control = make_event_control_matrices(X)
        return (predict_classifier(trained_model, X_event) - predict_classifier(trained_model, X_control)).astype(float)
    raise ValueError(f"Unknown model_family={model_family}")


def get_model_target_score_fn(model_bundle, model_name, target_name):
    model_family = get_base_model_family(model_name)
    trained_model = model_bundle[model_name][target_name]
    def score_fn(X):
        X = np.asarray(X, dtype=np.float32)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        return predict_trained_model_uplift(trained_model, model_family, X)
    return score_fn


def sample_rows(X, n_rows, seed):
    rng = np.random.default_rng(seed)
    n = len(X)
    size = min(int(n_rows), n)
    idx = rng.choice(n, size=size, replace=False)
    return X[idx], idx


def compute_lime_importance(score_fn, X_background, X_explain, feature_names, seed):
    explainer = LimeTabularExplainer(
        training_data=np.asarray(X_background, dtype=np.float32),
        feature_names=list(feature_names),
        mode="regression",
        discretize_continuous=True,
        random_state=seed,
    )
    rng = np.random.default_rng(seed)
    row_idx = rng.choice(len(X_explain), size=min(LIME_SAMPLE_SIZE, len(X_explain)), replace=False)
    importance = np.zeros(len(feature_names), dtype=float)
    for i in row_idx:
        exp = explainer.explain_instance(
            data_row=np.asarray(X_explain[i], dtype=np.float32),
            predict_fn=score_fn,
            num_features=len(feature_names),
        )
        label = next(iter(exp.local_exp.keys()))
        for feature_idx, weight in exp.local_exp[label]:
            importance[int(feature_idx)] += abs(float(weight))
    return importance / max(len(row_idx), 1)


def compute_shap_importance(score_fn, X_background, X_explain):
    explainer = shap.KernelExplainer(score_fn, np.asarray(X_background, dtype=np.float32))
    try:
        shap_values = explainer.shap_values(np.asarray(X_explain, dtype=np.float32), nsamples=SHAP_NSAMPLES, silent=True)
    except TypeError:
        shap_values = explainer.shap_values(np.asarray(X_explain, dtype=np.float32), nsamples=SHAP_NSAMPLES)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.asarray(shap_values, dtype=float)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 0]
    return np.mean(np.abs(shap_values), axis=0)


def qini_total_from_scores(y_cash, y_card, w_cash, w_card, cash_score, card_score):
    q_cash = qini_metrics(y_cash, w_cash, cash_score)["qini_norm"]
    q_card = qini_metrics(y_card, w_card, card_score)["qini_norm"]
    q_cash = 0.0 if pd.isna(q_cash) else float(q_cash)
    q_card = 0.0 if pd.isna(q_card) else float(q_card)
    return q_cash + q_card


def compute_permutation_importance(cash_score_fn, card_score_fn, X_eval, y_cash, y_card, W_eval, seed):
    rng = np.random.default_rng(seed)
    X_eval = np.asarray(X_eval, dtype=np.float32)
    y_cash = np.asarray(y_cash, dtype=int)
    y_card = np.asarray(y_card, dtype=int)
    W_eval = np.asarray(W_eval, dtype=int)
    w_cash = (W_eval == 1).astype(int)
    w_card = (W_eval == 0).astype(int)
    baseline_cash = cash_score_fn(X_eval)
    baseline_card = card_score_fn(X_eval)
    baseline = qini_total_from_scores(y_cash, y_card, w_cash, w_card, baseline_cash, baseline_card)
    importance = np.zeros(X_eval.shape[1], dtype=float)
    for j in range(X_eval.shape[1]):
        drops = []
        for _ in range(PERMUTATION_REPEATS):
            X_perm = X_eval.copy()
            X_perm[:, j] = rng.permutation(X_perm[:, j])
            perm_cash = cash_score_fn(X_perm)
            perm_card = card_score_fn(X_perm)
            perm_score = qini_total_from_scores(y_cash, y_card, w_cash, w_card, perm_cash, perm_card)
            drops.append(baseline - perm_score)
        importance[j] = float(np.mean(drops))
    return importance


def compute_train_qini_table(final_pred, X_train):
    rows = []
    for model_name in final_pred["base_model_names"]:
        cash_score_fn = get_model_target_score_fn(final_pred["models"], model_name, "cash")
        card_score_fn = get_model_target_score_fn(final_pred["models"], model_name, "card")
        cash_score = cash_score_fn(X_train)
        card_score = card_score_fn(X_train)
        qini_cash = qini_metrics(Y_cash_all, (W_all == 1).astype(int), cash_score)["qini_norm"]
        qini_card = qini_metrics(Y_card_all, (W_all == 0).astype(int), card_score)["qini_norm"]
        rows.append({
            "model": model_name,
            "qini_cash": qini_cash,
            "qini_card": qini_card,
            "qini_sum": qini_cash + qini_card,
        })
    ensemble_cash = np.mean(np.column_stack([
        get_model_target_score_fn(final_pred["models"], name, "cash")(X_train)
        for name in final_pred["base_model_names"]
    ]), axis=1)
    ensemble_card = np.mean(np.column_stack([
        get_model_target_score_fn(final_pred["models"], name, "card")(X_train)
        for name in final_pred["base_model_names"]
    ]), axis=1)
    qini_cash = qini_metrics(Y_cash_all, (W_all == 1).astype(int), ensemble_cash)["qini_norm"]
    qini_card = qini_metrics(Y_card_all, (W_all == 0).astype(int), ensemble_card)["qini_norm"]
    rows.append({"model": ENSEMBLE_NAME, "qini_cash": qini_cash, "qini_card": qini_card, "qini_sum": qini_cash + qini_card})
    return pd.DataFrame(rows)


def compute_and_save_feature_importance(final_pred, X_train, encoded_feature_names):
    X_importance, importance_idx = sample_rows(X_train, IMPORTANCE_SAMPLE_SIZE, SEED + 1000)
    y_cash_imp = Y_cash_all[importance_idx]
    y_card_imp = Y_card_all[importance_idx]
    W_imp = W_all[importance_idx]
    X_background, _ = sample_rows(X_importance, SHAP_BACKGROUND_SIZE, SEED + 2000)
    X_shap, _ = sample_rows(X_importance, SHAP_SAMPLE_SIZE, SEED + 3000)
    all_detail = []
    for model_idx, model_name in enumerate(final_pred["base_model_names"], start=1):
        print("=" * 80)
        print(f"FEATURE IMPORTANCE MODEL {model_idx}: {model_name}")
        print("=" * 80)
        cash_score_fn = get_model_target_score_fn(final_pred["models"], model_name, "cash")
        card_score_fn = get_model_target_score_fn(final_pred["models"], model_name, "card")
        lime_cash = compute_lime_importance(cash_score_fn, X_background, X_importance, encoded_feature_names, SEED + model_idx * 10 + 1)
        lime_card = compute_lime_importance(card_score_fn, X_background, X_importance, encoded_feature_names, SEED + model_idx * 10 + 2)
        shap_cash = compute_shap_importance(cash_score_fn, X_background, X_shap)
        shap_card = compute_shap_importance(card_score_fn, X_background, X_shap)
        permutation_importance = compute_permutation_importance(cash_score_fn, card_score_fn, X_importance, y_cash_imp, y_card_imp, W_imp, SEED + model_idx * 10 + 3)
        detail = pd.DataFrame({
            "feature": encoded_feature_names,
            "lime_importance": (lime_cash + lime_card) / 2.0,
            "shap_importance": (shap_cash + shap_card) / 2.0,
            "permutation_importance": permutation_importance,
        })
        detail = detail.sort_values("permutation_importance", ascending=False).reset_index(drop=True)
        detail_path = FEATURE_IMPORTANCE_DIR / f"model{model_idx}_feature_importance_detail.csv"
        detail.to_csv(detail_path, index=False)
        detail_for_summary = detail.copy()
        detail_for_summary["model"] = model_name
        all_detail.append(detail_for_summary)
        print("Saved:", detail_path)
        display(detail.head(20))
        gc.collect()
    stacked = pd.concat(all_detail, ignore_index=True)
    summary = stacked.groupby("feature", as_index=False)[["lime_importance", "shap_importance", "permutation_importance"]].mean()
    summary["feature_importance"] = summary[["lime_importance", "shap_importance", "permutation_importance"]].mean(axis=1)
    summary = summary.sort_values("feature_importance", ascending=False).reset_index(drop=True)
    summary_path = FEATURE_IMPORTANCE_DIR / "feature_importance_detail.csv"
    summary.to_csv(summary_path, index=False)
    print("=" * 80)
    print("FEATURE IMPORTANCE SUMMARY SAVED")
    print("Saved:", summary_path)
    print("=" * 80)
    display(summary.head(30))
    return summary


print("FEATURE IMPORTANCE FUNCTIONS READY")


FEATURE IMPORTANCE FUNCTIONS READY


## 9. Full-train, inference test và export output

In [ ]:
print("=" * 80)
print("START FULL-TRAIN FIT + TEST INFERENCE")
print("=" * 80)

X_full, [X_test], final_encoded_cols = make_design_matrices(
    fit_feature_df=df_feature,
    transform_feature_dfs=[test_df_feature],
    feat_cols=FEAT_COLS,
)

final_pred = train_ensemble_predict(
    X_fit=X_full,
    y_cash_fit=Y_cash_all,
    y_card_fit=Y_card_all,
    W_fit=W_all,
    X_pred=X_test,
    phase_label="full_train",
)

train_qini_table = compute_train_qini_table(final_pred, X_full)
train_qini_path = CP / "train_qini_by_model.csv"
train_qini_table.to_csv(train_qini_path, index=False)

print("=" * 80)
print("TRAIN QINI BY MODEL")
print(train_qini_table[["model", "qini_cash", "qini_card", "qini_sum"]].round(6).to_string(index=False))
print("Saved:", train_qini_path)
print("=" * 80)

display(train_qini_table[["model", "qini_cash", "qini_card", "qini_sum"]].round(6))

submission = pd.DataFrame({"ID": test_id.values})
for model_idx, model_name in enumerate(final_pred["base_model_names"], start=1):
    submission[f"model{model_idx}_cash_sign_30d"] = np.clip(final_pred["base_predictions"]["cash"]["prob_event"][model_name], 0.0, 1.0)
    card_col = f"model{model_idx}_card_sign_30d" if model_idx != 5 else "model5card_sign_30d"
    submission[card_col] = np.clip(final_pred["base_predictions"]["card"]["prob_event"][model_name], 0.0, 1.0)
submission[TARGET_CASH] = np.nan
submission[TARGET_CARD] = np.nan

submission_path = CP / "submission.csv"
submission.to_csv(submission_path, index=False)

final_run_log_path = CP / "full_train_run_log.csv"
pd.DataFrame(final_pred["log_rows"]).to_csv(final_run_log_path, index=False)

encoded_columns_path = CP / "final_encoded_feature_columns.csv"
pd.DataFrame({"encoded_feature": final_encoded_cols}).to_csv(encoded_columns_path, index=False)

feature_importance_summary = compute_and_save_feature_importance(final_pred, X_full, final_encoded_cols)

print("=" * 80)
print("FULL-TRAIN INFERENCE DONE")
print("Saved:", submission_path)
print("Saved:", final_run_log_path)
print("Saved:", encoded_columns_path)
print("Feature importance folder:", FEATURE_IMPORTANCE_DIR)
print("=" * 80)
print("submission preview:")
display(submission.head())


START FULL-TRAIN FIT + TEST INFERENCE

full_train START CFG028_UPLIFT_RF_CIT
{'name': 'CFG028_UPLIFT_RF_CIT', 'n_estimators': 80, 'max_depth': 8, 'min_samples_leaf': 120, 'min_samples_treatment': 20, 'max_features_cap': 32}
full_train finished CFG028_UPLIFT_RF_CIT target=CASH | minutes=8.0842
full_train finished CFG028_UPLIFT_RF_CIT target=CARD | minutes=7.8653

full_train START CFG009_UPLIFT_RF_CIT
{'name': 'CFG009_UPLIFT_RF_CIT', 'n_estimators': 120, 'max_depth': 5, 'min_samples_leaf': 80, 'min_samples_treatment': 30, 'max_features_cap': 32}
full_train finished CFG009_UPLIFT_RF_CIT target=CASH | minutes=2.9185
full_train finished CFG009_UPLIFT_RF_CIT target=CARD | minutes=2.8265

full_train START CFG011_UPLIFT_RF_CIT
{'name': 'CFG011_UPLIFT_RF_CIT', 'n_estimators': 120, 'max_depth': 5, 'min_samples_leaf': 160, 'min_samples_treatment': 40, 'max_features_cap': 32}
full_train finished CFG011_UPLIFT_RF_CIT target=CASH | minutes=2.9093
full_train finished CFG011_UPLIFT_RF_CIT target=CARD 

,model,qini_cash,qini_card,qini_sum
0,CFG028_UPLIFT_RF_CIT,0.134977,0.232503,0.367480
1,CFG009_UPLIFT_RF_CIT,0.068274,0.069925,0.138199
2,CFG011_UPLIFT_RF_CIT,0.068305,0.069816,0.138122
3,INT_RF_RS_027,0.088286,0.144436,0.232722
4,INT_RF_RS_028,0.091589,0.167941,0.259530
5,ENSEMBLE_3_UPLIFT_RF_CIT_2_INT_RF,0.088185,0.158916,0.247102


FEATURE IMPORTANCE MODEL 1: CFG028_UPLIFT_RF_CIT


## 10. Export config và zip output

In [ ]:
param_rows = []
for cfg in UPLIFT_RF_CIT_CONFIGS:
    param_rows.append({"model_family": "UPLIFT_RF_CIT", **cfg})
for cfg in INT_RF_CONFIGS:
    param_rows.append({"model_family": "INT_RF", **cfg})

base_params_path = CP / "base_model_params.csv"
pd.DataFrame(param_rows).to_csv(base_params_path, index=False)

print("=" * 80)
print("ALL SAVED FILES IN CP")
for p in sorted(CP.rglob("*")):
    print(p)
print("=" * 80)

import shutil

zip_base = str(WORKING_DIR / "cp")
zip_path = shutil.make_archive(zip_base, "zip", CP)
print("ZIP SAVED:", zip_path)
print("=" * 80)
print("DONE")
print("=" * 80)

In [ ]:
!zip -r /kaggle/working/cp_final.zip /kaggle/working/cp